# Projet DS — Exploration du jeu de données MVTec AD

**Détection d'anomalies (approche non-supervisée)**

Dans ce notebook, on reproduit en Python les graphiques que nous avions d'abord faits sous Excel.
On part d'un fichier `mvtec_manifest.csv` qui contient **une ligne par image** du dataset, avec :
la catégorie, le sous-ensemble (train/test), le label, si c'est un défaut ou non, la résolution et le mode couleur.

Plan :
1. Charger les données
2. Composition des images par catégorie
3. Pourcentage de défauts dans le test
4. Modes couleur (RGB / niveaux de gris)
5. Nombre de types de défauts par catégorie
6. Résolutions des images

## 1. Importer les librairies et charger les données

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# On charge le manifeste (une ligne = une image)
df = pd.read_csv("../data/mvtec_manifest.csv")

print("Nombre d'images :", len(df))
df.head()

On regarde rapidement le contenu pour comprendre les colonnes.

In [ ]:
df.info()

## 2. Quelques chiffres généraux

On compte le nombre d'images de train, de test, etc.

In [ ]:
nb_train = len(df[(df["split"] == "train") & (df["label"] == "good")])
nb_test = len(df[df["split"] == "test"])
nb_test_good = len(df[(df["split"] == "test") & (df["label"] == "good")])
nb_test_defaut = len(df[(df["split"] == "test") & (df["is_defect"] == 1)])

print("Nombre de catégories :", df["category"].nunique())
print("Images de train (normales) :", nb_train)
print("Images de test :", nb_test)
print("  - test normales (good) :", nb_test_good)
print("  - test défectueuses :", nb_test_defaut)
print("Pourcentage de défauts dans le test :", round(nb_test_defaut / nb_test * 100, 1), "%")

## 3. Graphique 1 — Composition des images par catégorie

Pour chaque catégorie, on veut 3 chiffres : les images de train, les images de test normales,
et les images de test défectueuses. On les calcule puis on fait un graphique en barres empilées.

In [ ]:
# On prépare un tableau avec une ligne par catégorie
categories = sorted(df["category"].unique())

train_par_cat = []
test_good_par_cat = []
test_defaut_par_cat = []

for cat in categories:
    sous_df = df[df["category"] == cat]
    train_par_cat.append(len(sous_df[(sous_df["split"] == "train") & (sous_df["label"] == "good")]))
    test_good_par_cat.append(len(sous_df[(sous_df["split"] == "test") & (sous_df["label"] == "good")]))
    test_defaut_par_cat.append(len(sous_df[(sous_df["split"] == "test") & (sous_df["is_defect"] == 1)]))

# On range tout dans un DataFrame, plus pratique
compo = pd.DataFrame({
    "categorie": categories,
    "train": train_par_cat,
    "test_good": test_good_par_cat,
    "test_defaut": test_defaut_par_cat,
})
compo

In [ ]:
plt.figure(figsize=(12, 6))

# Barres empilées : on empile train, puis test_good, puis test_defaut
plt.bar(compo["categorie"], compo["train"], label="Train (normales)", color="steelblue")
plt.bar(compo["categorie"], compo["test_good"], bottom=compo["train"],
        label="Test normales", color="mediumseagreen")
plt.bar(compo["categorie"], compo["test_defaut"],
        bottom=compo["train"] + compo["test_good"],
        label="Test défauts", color="indianred")

plt.title("Composition des images par catégorie (train / test)")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/01_composition_par_categorie.png")
plt.show()

## 4. Graphique 2 — Pourcentage de défauts dans le test

On calcule, pour chaque catégorie, la part d'images défectueuses parmi les images de test.

In [ ]:
compo["pct_defaut"] = compo["test_defaut"] / (compo["test_good"] + compo["test_defaut"]) * 100
compo_tri = compo.sort_values("pct_defaut", ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(compo_tri["categorie"], compo_tri["pct_defaut"], color="indianred")
plt.title("% de défauts dans le jeu de test, par catégorie")
plt.ylabel("% de défauts")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../figures/02_pct_defauts_test.png")
plt.show()

## 5. Graphique 3 — Modes couleur

Certaines catégories sont en couleur (RGB), d'autres en niveaux de gris (L).
On compte le nombre de catégories pour chaque mode.

In [ ]:
# Une seule valeur de mode par catégorie : on enlève les doublons
mode_par_cat = df.drop_duplicates("category")
nb_par_mode = mode_par_cat["mode"].value_counts()
print(nb_par_mode)

plt.figure(figsize=(6, 6))
plt.pie(nb_par_mode, labels=nb_par_mode.index, autopct="%1.0f%%",
        colors=["steelblue", "lightgray"])
plt.title("Modes couleur (RGB vs niveaux de gris)")
plt.savefig("../figures/03_modes_couleur.png")
plt.show()

## 6. Graphique 4 — Nombre de types de défauts par catégorie

On compte combien de types de défauts différents existent pour chaque catégorie.

In [ ]:
defauts = df[df["is_defect"] == 1]
nb_types = defauts.groupby("category")["label"].nunique()
nb_types = nb_types.sort_values(ascending=False)
print(nb_types)

plt.figure(figsize=(12, 6))
plt.bar(nb_types.index, nb_types.values, color="teal")
plt.title("Nombre de types de défauts par catégorie")
plt.ylabel("Nombre de types de défauts")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../figures/04_nb_types_defauts.png")
plt.show()

## 7. Graphique 5 — Résolutions des images

On regarde combien de catégories ont chaque résolution.

In [ ]:
res_par_cat = df.drop_duplicates("category")
nb_par_res = res_par_cat["resolution"].value_counts()
print(nb_par_res)

plt.figure(figsize=(10, 6))
plt.bar(nb_par_res.index, nb_par_res.values, color="darkorange")
plt.title("Hétérogénéité des résolutions (nb de catégories)")
plt.ylabel("Nombre de catégories")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../figures/05_resolutions.png")
plt.show()

## 8. Conclusion

- Le dataset contient 15 catégories et plus de 5000 images.
- Les images de **train** sont uniquement des pièces **normales** : c'est normal, le projet est en
  détection d'anomalies non-supervisée (on apprend la « normale » et on détecte les écarts).
- Le jeu de test est **déséquilibré** (beaucoup plus de défauts que de bonnes pièces) :
  il vaudra donc mieux utiliser des métriques comme l'AUROC ou le rappel plutôt que l'accuracy.
- Les catégories n'ont pas toutes la même résolution ni le même mode couleur :
  il faudra harmoniser les images avant la modélisation.